# Eole による英日ニューラル機械翻訳演習
Eole というツールを用いて英日機械翻訳の演習を行います。Eole は、書籍で紹介している OpenNMT-py の後継のツールです。OpenNMT-py は2024年の v3.5.1 で開発が止まっているため、この演習では Eole を使います。コマンド名や設定ファイルの書き方は少し違いますが、できることはほぼ同じです。
Google Colaboratory は GPU を用いた実験ができるので、GPU を用いてニューラル機械翻訳の実験をしましょう。（「ランタイム」→「ランタイムのタイプを変更」から「ハードウェア アクセラレータ」を GPU（T4 など）にしておいてください。以下、Python 3 で説明します。）
データは田中コーパスという自由に使える対訳コーパスを使用します。翻訳結果を実際に見て評価する、ということも演習の目的です。

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU が利用可能です: {torch.cuda.get_device_name(0)}")
else:
    print(
        "警告: GPU が検出されませんでした（現在 CPU モードです）。\n"
        "「ランタイム」→「ランタイムのタイプを変更」から「ハードウェア アクセラレータ」を GPU（T4 など）に変更してください。"
    )

# Eole のインストール

## pip によるインストール
Eole を使って実験を行います。開発が速く、数週間で内容が変わってしまうことがあるので、ここでは2026年10月時点の最新版である v0.6.4 というバージョンを使います。pip というのは Python のソフトウェアをインストールするときに使うツールの名前で、依存関係も含めて自動でセットアップしてくれます。

ただし、2026年10月時点の Google Colab の Python は 3.13 ですが、Eole が使うライブラリ（pyonmttok や fasttext）は Python 3.12 までしか対応していないため、そのままでは `pip install eole` が失敗します。そこで、[uv](https://docs.astral.sh/uv/) という高速なパッケージ管理ツールを使って Python 3.12 の仮想環境を /content/eole-env に作り、そこに Eole と PyTorch をインストールします。1〜2分かかります。最後に `True` と表示されれば GPU が使えます。

In [ ]:
!pip install -q uv
!uv venv -q --clear -p 3.12 /content/eole-env
!uv pip install -q -p /content/eole-env eole==0.6.4
!/content/eole-env/bin/python -c "import sys, torch; print(sys.version, torch.__version__, torch.cuda.is_available())"

以下では `!eole ...` のようにコマンドを実行するので、作った仮想環境のコマンドが使えるように PATH を設定しておきます。ランタイムを再起動した場合は、このセルをもう一度実行してください。

In [ ]:
import os
os.environ["PATH"] = "/content/eole-env/bin:" + os.environ["PATH"]
!which eole

# 対訳データのインストールと前処理

## 田中コーパス
田中コーパスというデータで翻訳実験を行います。日本の大学生が翻訳したデータです。そのうち適当な長さの5万文だけを抽出して実験しやすいようにしてくれたデータを用います。
train.{ja,en}というのが学習データ、dev.{ja,en}というのが開発データ、test.{ja,en}というのがテストデータです。

In [ ]:
![ -d small_parallel_enja ] || git clone https://github.com/odashi/small_parallel_enja

In [ ]:
!ls small_parallel_enja

## データの前処理
`eole build_vocab` というコマンドでデータの前処理をします。語彙のサイズをコントロールしたり、単語分割をしたりします。YAML という形式で書かれた設定ファイルが必要なので、準備しておきます。以下では tanaka-enja.yaml という名前で設定ファイルを保存します。
Eole の設定ファイルでは、モデルの設定を `model:`、学習の設定を `training:` の下にまとめて書きます。

In [ ]:
%%bash
cat > tanaka-enja.yaml << __EOF__
# 前処理したデータの出力先の設定
save_data: ./tanaka
src_vocab: ./tanaka/vocab.src
tgt_vocab: ./tanaka/vocab.tgt
overwrite: True

# 学習に必要なデータが置いてある置き場所
data:
    corpus_1:
        path_src: small_parallel_enja/train.en
        path_tgt: small_parallel_enja/train.ja
    valid:
        path_src: small_parallel_enja/dev.en
        path_tgt: small_parallel_enja/dev.ja

# モデルの設定（RNN (LSTM) を使う）
model:
    architecture: rnn

# ここから学習時の設定
training:
    world_size: 1
    gpu_ranks: [0]
    model_path: ./tanaka/run/model
    save_checkpoint_steps: 500
    train_steps: 20000
    valid_steps: 500
__EOF__

In [ ]:
!eole build_vocab -c tanaka-enja.yaml -n_sample 10000

# NMTの訓練

## 学習
`eole train` というコマンドで学習します。上述の設定ファイルで挙動を制御できます。いくつかオプションがあるので、変えてみると挙動も変わります。
world_size で使う GPU の個数、gpu_ranks で使う GPU を指定します（0スタートです）。指定しないと CPU になってしまいます。CPU は GPU の数十倍遅いです。
train_steps は学習するステップを指定します。20,000だとこのデータは T4 GPU で30分弱で学習できます（500ステップごとの検証を含みます）。

デフォルトでは `tanaka-enja.yaml`（`architecture: rnn` の RNN (LSTM)）を使って学習します。Transformer を試したい場合、RNN のデフォルト設定（SGD や文単位バッチ）のまま `architecture: transformer` だけを変更すると学習が収束しないため、`optim: adam`、`decay_method: noam`、`warmup_steps: 4000`、トークン単位バッチ（`batch_type: tokens`）、`label_smoothing: 0.1` などの設定を合わせて指定する必要があります。下のセルでは Transformer 用の設定ファイル `tanaka-enja-transformer.yaml` も生成しており、`USE_TRANSFORMER = True` に切り替えるだけで試せます。その他の設定は[設定ファイルの書き方](https://eole-nlp.github.io/eole/docs/concepts/config)や[モデルの設定一覧](https://eole-nlp.github.io/eole/docs/reference/Config/models)を見て編集してみましょう。（ただし、今回はデータが少ないので Transformer で学習しても必ずしも RNN より精度が高くなるとは限りません）

In [ ]:
import os
from pathlib import Path

USE_TRANSFORMER = False  # Transformer を試す場合は True に変更

transformer_yaml = """\
# 前処理したデータの出力先の設定
save_data: ./tanaka
src_vocab: ./tanaka/vocab.src
tgt_vocab: ./tanaka/vocab.tgt
overwrite: True

# 学習に必要なデータの置き場所
data:
    corpus_1:
        path_src: small_parallel_enja/train.en
        path_tgt: small_parallel_enja/train.ja
    valid:
        path_src: small_parallel_enja/dev.en
        path_tgt: small_parallel_enja/dev.ja

# モデルの設定（Transformer を使う）
model:
    architecture: transformer
    layers: 6
    heads: 8
    hidden_size: 512
    transformer_ff: 2048
    embeddings:
        word_vec_size: 512
        position_encoding_type: SinusoidalInterleaved

# 学習時の設定（Transformer 向けの代表的なハイパーパラメータ）
training:
    world_size: 1
    gpu_ranks: [0]
    model_path: ./tanaka/run/model
    save_checkpoint_steps: 500
    train_steps: 20000
    valid_steps: 500
    optim: adam
    adam_beta2: 0.998
    learning_rate: 2
    decay_method: noam
    warmup_steps: 4000
    batch_type: tokens
    batch_size: 2048
    normalization: tokens
    max_grad_norm: 0
    dropout: [0.1]
    attention_dropout: [0.1]
    label_smoothing: 0.1
    param_init_method: xavier_uniform
"""

Path("tanaka-enja-transformer.yaml").write_text(transformer_yaml, encoding="utf-8")

config_file = "tanaka-enja-transformer.yaml" if USE_TRANSFORMER else "tanaka-enja.yaml"
os.environ["EOLE_CONFIG"] = config_file
print(f"使用する設定ファイル: {config_file}")

In [ ]:
!eole train -c "${EOLE_CONFIG:-tanaka-enja.yaml}"

## 翻訳モデル
学習されたモデルは tanaka/run/model の下に step_10000 のような名前のディレクトリで保存されています。tanaka/run/model 自体を指定すると、最後に保存されたモデルが使われます。

In [ ]:
!ls tanaka/run/model

# NMTによる翻訳

## デコード
`eole predict` というコマンドでデコードします。`--model_path` で一つ前に学習した翻訳モデルのうちの1つを指定します。
`--src` で翻訳したい入力ファイル、`--output pred.txt` で結果を保存するファイルを指定します。
`--gpu_ranks 0` で GPU を使って翻訳します。
`--verbose` は結果を冗長に画面にも表示させるオプションです。

In [ ]:
!eole predict --model_path tanaka/run/model/step_20000 --src small_parallel_enja/test.en --output pred.txt --gpu_ranks 0 --verbose

## BLEU による評価
正解（参照訳）のファイルを第一引数に、システムの出力をリダイレクト（<）で渡します。
sacrebleu というツールで評価します。Eole と一緒に仮想環境にインストールされているので、そのまま使えます。

> **注意（日本語のトークナイズについて）**: `small_parallel_enja/test.ja` はあらかじめ分かち書き（スペース区切り）されたデータのため、ここではそのまま `sacrebleu` でスペース区切りトークン単位の BLEU を計算しています。分かち書きされていない通常の日本語テキストを評価する場合は、`sacrebleu --tokenize ja-mecab` のように日本語向けトークナイザを指定してください（詳細は `ch4_smt_bleu.ipynb` を参照してください）。

In [ ]:
!sacrebleu small_parallel_enja/test.ja < pred.txt

## モデル選択

どのモデルがいいかというのは未知のデータで評価する必要があります。ここでは開発データとテストデータが用意されているので、開発データでもっとも性能がよいモデルを自動で選び、テストデータの翻訳を行います。
下のセルでは 2,000 ステップごと（計10チェックポイント）に開発データを翻訳して BLEU を計測し、学習曲線（Step vs Dev BLEU）を描画したうえで最良のステップ `BEST_STEP` を自動選択します（より細かく 1,000 ステップ刻みで見たい場合はコード内の `range` を変更してください）。
参考として、2026年10月に試したときは開発データでの BLEU は 20000 ステップのモデルが 35.4 で一番高く、次いで 16000 ステップのモデルが 35.1 でした。学習のたびに結果は少し変わります。

In [ ]:
import os
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt

# 2,000ステップ刻みで評価（1,000ステップ刻みにしたい場合は range(1000, 20001, 1000) に変更）
steps = list(range(2000, 20001, 2000))
results = []

for step in steps:
    model_dir = Path(f"tanaka/run/model/step_{step}")
    if not model_dir.exists():
        print(f"step {step}: チェックポイントが見つからないためスキップします ({model_dir})")
        continue
    pred_path = Path(f"pred_step_{step}.txt")
    subprocess.run(
        [
            "eole", "predict",
            "--model_path", str(model_dir),
            "--src", "small_parallel_enja/dev.en",
            "--output", str(pred_path),
            "--gpu_ranks", "0",
        ],
        check=True,
    )
    with pred_path.open("r", encoding="utf-8") as f:
        bleu_out = subprocess.run(
            ["sacrebleu", "-b", "small_parallel_enja/dev.ja"],
            stdin=f,
            capture_output=True,
            text=True,
            check=True,
        )
    score = float(bleu_out.stdout.strip())
    results.append((step, score))
    print(f"step {step:5d}: Dev BLEU = {score:.2f}")

if not results:
    raise RuntimeError("評価可能なチェックポイントが見つかりませんでした。先に学習セルを実行してください。")

best_step, best_bleu = max(results, key=lambda x: x[1])
os.environ["BEST_STEP"] = str(best_step)
print(f"\n最良モデル: step_{best_step} (Dev BLEU = {best_bleu:.2f})")

eval_steps, eval_scores = zip(*results)
plt.figure(figsize=(8, 4))
plt.plot(eval_steps, eval_scores, marker="o", label="Dev BLEU")
plt.scatter([best_step], [best_bleu], color="red", zorder=5, label=f"Best: step {best_step} ({best_bleu:.2f})")
plt.xlabel("Training Step")
plt.ylabel("Dev BLEU")
plt.title("Dev BLEU Learning Curve")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
!echo "Best step: ${BEST_STEP}"
!eole predict --model_path "tanaka/run/model/step_${BEST_STEP}" --src small_parallel_enja/test.en --output pred.txt --gpu_ranks 0
!sacrebleu small_parallel_enja/test.ja < pred.txt

## 結果の確認と保存

テストデータの翻訳結果を `pandas.DataFrame` にまとめ、文単位の BLEU スコア（`sacrebleu.sentence_bleu`）を計算して翻訳の傾向を確認します。先頭10文に加えて、文BLEUが低いワースト10文や未知語（`<unk>`）の出現率も確認できます（生のタブ区切りで全件見たい場合は `!paste pred.txt small_parallel_enja/test.{ja,en}` も使えます）。
結果をファイルに保存する場合は `files.download` というメソッドを呼びます。
出力されていない結果は24時間で消えるので、ご注意ください。

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

try:
    import sacrebleu
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"], check=True)
    import sacrebleu

src_lines = Path("small_parallel_enja/test.en").read_text(encoding="utf-8").splitlines()
ref_lines = Path("small_parallel_enja/test.ja").read_text(encoding="utf-8").splitlines()
pred_lines = Path("pred.txt").read_text(encoding="utf-8").splitlines()

assert len(src_lines) == len(ref_lines) == len(pred_lines), (
    f"行数が一致しません: src={len(src_lines)}, ref={len(ref_lines)}, pred={len(pred_lines)}"
)

sent_bleus = [
    round(sacrebleu.sentence_bleu(pred, [ref]).score, 2)
    for pred, ref in zip(pred_lines, ref_lines)
]

df = pd.DataFrame({
    "原文 (en)": src_lines,
    "参照訳 (ja)": ref_lines,
    "システム訳 (pred)": pred_lines,
    "文BLEU": sent_bleus,
})

# <unk> の出現率を集計
unk_sent_count = sum("<unk>" in line.split() for line in pred_lines)
all_tokens = [tok for line in pred_lines for tok in line.split()]
unk_token_count = sum(tok == "<unk>" for tok in all_tokens)
print(f"テスト文数: {len(df)}")
print(f"<unk> を含む文の割合: {unk_sent_count}/{len(df)} ({unk_sent_count / len(df):.1%})")
print(f"<unk> トークンの割合: {unk_token_count}/{len(all_tokens)} ({unk_token_count / max(len(all_tokens), 1):.2%})")

pd.set_option("display.max_colwidth", 80)

print("\n=== 先頭 10 文の翻訳結果 ===")
display(df.head(10))

print("\n=== 文BLEU が低いワースト 10 文 ===")
display(df.sort_values("文BLEU", ascending=True).head(10))

In [ ]:
from google.colab import files
files.download("pred.txt")